In [1]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any


In [2]:
ds_plannerStudy_628 = pd.read_json(
    "Original datasets/plannerStudy_628.json", lines=True
)

In [28]:
samples = []
for row in ds_plannerStudy_628.index:
    samples.append({f"index_{row}": ds_plannerStudy_628["output"][row]})

In [38]:
# Compile all your day-name regex formats
DAYNAME_PATTERNS = [
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)(?:\s*-\s*(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday))?\s*\(Week\s*\d+\)(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*\(Week\s*\d+\):\s*(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday),\s*Week\s*\d+(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*-\s*Week\s*\d+(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?Week\s*\d+:\s*(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*(.*)$",
        re.IGNORECASE,
    ),
]


def contains_dayname_format(section_name: str) -> bool:
    """Check if a section name matches any of the day-name formats."""
    return any(p.match(section_name) for p in DAYNAME_PATTERNS)


def remove_dayname_plans(samples):
    filtered_samples = []

    for i, sample in enumerate(samples):
        try:
            sample_key = next(iter(sample.keys()))
        except StopIteration:
            print(f"Warning: Empty sample at position {i}. Skipping.")
            continue

        raw = sample[sample_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            print(f"Warning: Sample '{sample_key}' contains invalid JSON. Skipping.")
            continue

        sections = parsed.get("sections", [])
        has_dayname_format = any(
            contains_dayname_format(sec.get("name", "")) for sec in sections
        )

        if not has_dayname_format:
            filtered_samples.append(sample)

    return filtered_samples


def reindex_samples(samples):
    """
    Reindex the dataset so each sample has a fresh sequential index.
    Input: list of dicts like [{'index_327': json_string}, ...]
    Output: list of dicts like [{'index_0': json_string}, {'index_1': json_string}, ...]
    """
    new_samples = []
    for new_idx, sample in enumerate(samples):
        # Each sample has only one key like "index_327"
        old_key = list(sample.keys())[0]
        value = sample[old_key]
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: value})
    return new_samples


print(f"original length:{len(samples)}")
filtered_samples =remove_dayname_plans(samples)
filtered_samples=reindex_samples(filtered_samples)
print(f"filtered length:{len(filtered_samples)}")
# pprint(samples[0])
print("===========")
# pprint(filtered_samples[316])

original length:628
filtered length:602


In [44]:
week_sections_summary = []
count=0
for i, sample in enumerate(filtered_samples):
    key = f"index_{i}"
    try :
        raw = sample[key]
    except KeyError:
        count+=1
        print(f"Sample {i} does not have key '{key}'")
        continue

    try:
        parsed = json.loads(raw)  # convert JSON string → dict
    except json.JSONDecodeError as e:
        print(f"Error in sample {i}: {e}")
        print("Raw string:", raw[:200], "...")  # preview first 200 chars
        continue  # skip invalid JSON

    plan_name = parsed.get("name")
    sections = parsed.get("sections", [])

    # Filter sections containing "week" (case-insensitive)
    week_sections = [
        sec
        for sec in sections
        # if contains_dayname_format(sec.get("name",""))
        if re.search(
            r"\b(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday): \b",
            sec["name"],
            re.IGNORECASE,
        )
    ]

    if week_sections:
        week_sections_summary.append(
            {
                "plan_name": plan_name,
                "count": len(week_sections),
                "sections": week_sections,
            }
        )

# Print results
for summary in week_sections_summary:
    print(f"Plan: {summary['plan_name']}")
    print(f"\tNumber of 'week' sections: {summary['count']}\n")
    for sec in summary["sections"]:
        print(f"\tSection: {sec['name']}")
        print(f"\tOutline: {sec['outline']}")
        # print("Resources:")
        # for res in sec.get("resources", []):
        #     print(f"  - {res['type']}: {', '.join(res['keywords'])}")
        print()
print(f"Total plans with 'week' sections: {len(week_sections_summary)}")
print (f"count",count)

Plan: AP Environmental Science: Microplastic Pollution Study Plan
	Number of 'week' sections: 8

	Section: Saturday: Introduction and Overview
	Outline: Start with an overview of microplastic pollution. Watch an introductory video to understand the basics. Use the Pomodoro technique to stay focused: 25 minutes of study followed by a 5-minute break. After the video, review flashcards to familiarize yourself with key terms.

	Section: Sunday: Sources and Types of Microplastics
	Outline: Focus on the sources and types of microplastics. Watch a detailed video on how microplastics enter marine environments. Use spaced repetition to review flashcards on different types of microplastics. End the day with a quiz to test your understanding.

	Section: Monday: Effects on Marine Biodiversity
	Outline: Since you're really busy today, focus on a short but comprehensive video about the effects of microplastics on marine biodiversity. Use any spare time to review flashcards on the topic.

	Section: T

In [ ]:
def expand_week_sections(plan: dict) -> dict:
    """
    Replace sections with names like 'Week 1: ...' into 5 daily tasks + 2 break days.
    Returns a new plan dict with expanded sections.
    """
    new_sections = []
    try:
        plan = json.loads(plan)
    except json.JSONDecodeError:
        print("Failed to parse plan as JSON")
        return plan  # Return original plan if parsing fails
    # print(plan.values())
    # plan=plan.values()
    for sec in plan.get("sections", []):
        try:
            name = sec.get("name", "")
            print(name)
        except Exception as e:
            print(
                f"Error processing section in plan {plan.get('name', 'Unknown')}: {e}"
            )
            continue
        match = re.match(r"(Week\s*\d+):\s*(.*)", name, flags=re.IGNORECASE)
        if match:
            week_label, base_title = match.groups()
            # Create 5 study days
            for i in range(1, 6):
                new_sections.append(
                    {
                        "index": i - 1,
                        "task": f"{base_title} - Day {i}",
                        "description": sec.get("outline", ""),
                        "is_repeatable": False,
                        "repeat_pattern": "none",
                        "estimated_duration_minutes": {"min": 30, "max": 45},
                        "gap_flag": False,
                    }
                )
            # Add 2 break days
            for i in range(1, 3):
                new_sections.append(
                    {
                        "index": i - 1,
                        "task": f"Break Day",
                        "description": "Take a rest day to recharge and consolidate learning.",
                        "is_repeatable": False,
                        "repeat_pattern": "none",
                        "estimated_duration_minutes": {"min": 0, "max": 0},
                        "gap_flag": True,
                    }
                )
        else:
            # Keep non-week sections unchanged
            new_sections.append(sec)

    # Return updated plan
    return {**plan, "sections": new_sections}


s = expand_week_sections(samples[624]["index_624"])
pprint([sec.get("task") for sec in s.get("sections", [])])

In [ ]:
import json
import pandas as pd


def convert_row_to_schema(row):
    """
    Convert a single row of study plan data into the target schema.
    Assumes row contains keys like: course, unit, timeuntiltest, difficulty, busylevels, etc.
    """
    # Build baseline constraints
    baseline = {
        "fixed_constraints": f"Test scheduled in {row.get('timeuntiltest', 'N/A')}",
        "physical_constraints": f"Busy schedule: {row.get('busylevels', 'N/A')}",
        "non_negotiables": f"Must cover all unit topics for {row.get('unit', '')}",
    }

    # Build tasks from sections (if provided)
    tasks = []
    if "sections" in row:
        for i, section in enumerate(row["sections"]):
            tasks.append(
                {
                    "index": i,
                    "task": section.get("name", ""),
                    "description": section.get("outline", ""),
                    "is_repeatable": False,
                    "repeat_pattern": "none",
                    "estimated_duration_minutes": {
                        "min": 45,  # default estimate
                        "max": 90,  # default estimate
                    },
                }
            )

    schema = {
        "goal": f"Prepare for {row.get('course', '')} test on {row.get('unit', '')}",
        "goal_category": "one_time",
        "goal_type": "academic study",
        "time_horizon": row.get("timeuntiltest", ""),
        "description": f"A study plan for {row.get('course', '')} focusing on {row.get('unit', '')}.",
        "baseline": baseline,
        "tasks": tasks,
    }

    return schema


# Example: converting a dataset of 1k rows
def convert_dataset(input_file, output_file):
    """
    Reads a JSON or CSV dataset of study plans and converts each row.
    """
    # Load dataset (assuming JSON lines or CSV)
    if input_file.endswith(".json"):
        with open(input_file, "r") as f:
            data = json.load(f)
    else:
        df = pd.read_csv(input_file)
        data = df.to_dict(orient="records")

    # Convert each row
    converted = [convert_row_to_schema(row) for row in data]

    # Save output
    with open(output_file, "w") as f:
        json.dump(converted, f, indent=2)


# Example usage:
# convert_dataset("study_plans.json", "converted_schema.json")
# convert_dataset("study_plans.csv", "converted_schema.json")